In [ ]:
"""
Build a chatbot for the Indian Penal Code. We'll start by downloading the
official Indian Penal Code document, and then we'll create a chatbot that
can interact with it. Users will be able to ask questions about the
Indian Penal Code and have a conversation with it.
"""

In [ ]:
!pip install pymupdf faiss-cpu sentence-transformers

In [ ]:
import fitz
import faiss
import numpy as np
import re #Python’s regular expressions module.
from sentence_transformers import SentenceTransformer

In [ ]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_path = r"C:\Users\prajw\Gen AI\ipc_penal_code.pdf"
text = extract_text(pdf_path)

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text) # whitespace character
    return text

text = clean_text(text)

In [ ]:
def create_chunks(text, chunk_size=500):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        # extend to sentence boundary
        if end < len(text):
            while end < len(text) and text[end] not in ['.', '\n']:
                end += 1

        chunk = text[start:end].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        start = end

    return chunks

chunks = create_chunks(text)

print("Total chunks:", len(chunks))

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype('float32')

In [ ]:
dimension = embeddings.shape[1] # # Get size of each embedding vector to initialize FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


In [ ]:
def search(query, top_k=3):
    query = query.lower()

    q_embed = model.encode([query]).astype('float32')
    D, I = index.search(q_embed, top_k)

    print("\n Answer:\n")
    for i in I[0]:
        print("-", chunks[i], "\n")

In [ ]:
def chatbot():
    print("IPC Chatbot Ready! (type 'exit' to stop)\n")

    while True:
        query = input("You: ")

        if query.lower() == "exit":
            print("Bot: Goodbye!")
            break

        search(query)

In [ ]:
chatbot()